#TODO

Inspired by https://www.reddit.com/r/animation/comments/1u87tnu/playing_around_with_scribbles/ 

In [1]:
%matplotlib widget

In [2]:
import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backend_bases import MouseButton
import ipywidgets as widgets
from IPython.display import display

from PIL import Image
import vispy
# Force VisPy to use the Jupyter Remote FrameBuffer backend
from vispy import app, scene
from vispy.visuals.transforms import MatrixTransform
import imageio


vispy.use('pyqt6')
plt.rcParams["font.family"] = "monospace"

In [3]:
def aggressive_ease_in_out(x, power=7):
    """
    x: array or float from 0.0 to 1.0
    power: higher values = more time spent near 0 and 1, faster middle transition
    """
    # Prevent divide-by-zero edge case at x=0
    x = np.clip(x, 1e-9, 1.0 - 1e-9)
    
    xn = x ** power
    return xn / (xn + (1.0 - x) ** power)

# Example usage for 200 animation frames
linear_progress = np.linspace(0, 1, 200)
eased_progress = aggressive_ease_in_out(linear_progress, power=1.5)

def ease_out_in(x, power=3):
    """
    x: array or float from 0.0 to 1.0
    power: higher values = slower middle, more explosive start and end
    """
    # Shift input to be centered around 0, ranging from -1 to 1
    centered = 2 * x - 1
    
    # Apply a fractional power (root) to stretch the middle flat
    # We use np.sign to preserve the negative/positive sides
    eased = np.sign(centered) * (np.abs(centered) ** (1.0 / power))
    
    # Shift back to the 0.0 to 1.0 range
    return 0.5 * eased + 0.5

In [4]:
IMG_SIZE = 3.0

In [5]:
def draw_curve_widget():
    """Draw a 2D curve. This function returns the curve and the drawing time for each point."""
    plt.close('all')
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # Pre-allocated arrays
    draw_x = np.zeros(2000)
    draw_y = np.zeros(2000)
    draw_time = np.zeros(2000)
    
    img = plt.imread('paper_texture.png') 
    ax.imshow(img, extent=[-IMG_SIZE/2, IMG_SIZE/2, -IMG_SIZE/2, IMG_SIZE/2], aspect='auto')
    
    plot_draw = ax.plot(draw_x, draw_y, color='red', linewidth=3)
    text0 = ax.text(0, IMG_SIZE/2 + 0.1, 'Current position: (NaN, NaN)', horizontalalignment='center')
    
    UPDATE_INTERVAL = 30  
    last_update_time = datetime.datetime.now()
    start_time = datetime.datetime.now()
    num_pts = 0

    def on_move(event):
        nonlocal num_pts, last_update_time, draw_x, draw_y, draw_time, plot_draw, text0, fig
        if event.inaxes and max(np.abs(event.xdata), np.abs(event.ydata)) <= 1.0:  
            if event.button is MouseButton.LEFT:
                if num_pts >= len(draw_x):
                    return
                draw_x[num_pts] = event.xdata
                draw_y[num_pts] = event.ydata
                draw_time[num_pts] = (datetime.datetime.now() - start_time).total_seconds() * 1000
                num_pts += 1
                
                if (datetime.datetime.now() - last_update_time).total_seconds() * 1000 > UPDATE_INTERVAL:
                    plot_draw[0].set_data(draw_x[:num_pts], draw_y[:num_pts])

            if (datetime.datetime.now() - last_update_time).total_seconds() * 1000 > UPDATE_INTERVAL:
                text0.set_text(f'Current position: ({event.xdata: .2f}, {event.ydata: .2f})')
                fig.canvas.draw_idle()  # Use draw_idle for smoother widget rendering
                last_update_time = datetime.datetime.now()

    def on_click(event):
        nonlocal num_pts, last_update_time, plot_draw, fig
        if event.button is MouseButton.LEFT:
            num_pts = 0
            plot_draw[0].set_data([], [])
            fig.canvas.draw_idle()
            last_update_time = datetime.datetime.now()

    ax.set_aspect('equal')
    ax.set_xlim([-IMG_SIZE/2, IMG_SIZE/2])
    ax.set_ylim([-IMG_SIZE/2, IMG_SIZE/2])
    ax.set_axis_off()

    motion_binding_id = plt.connect('motion_notify_event', on_move)
    click_binding_id = plt.connect('button_press_event', on_click)

    # --- Create a Jupyter UI Button ---
    btn_finish = widgets.Button(description="Finish & Export Curve", button_style='success')
    output_area = widgets.Output()

    def on_finish_clicked(_):
        """Callback that safely executes only when the user clicks 'Finish'."""
        with output_area:
            output_area.clear_output()
            
            # Safely slice data now that drawing is complete
            final_pts_2d = np.column_stack((draw_x[:num_pts], draw_y[:num_pts]))
            final_draw_t = draw_time[:num_pts].copy()
            
            print(f"Drawing finished successfully! Collected {num_pts} points.")
            
            np.savez('curve_2d.npz', pts_2d=final_pts_2d, draw_t=final_draw_t)
            print("Saved to 'curve_2d.npz'!")
                
            # Disconnect event bindings to freeze the plot interactions
            plt.disconnect(motion_binding_id)
            plt.disconnect(click_binding_id)
            btn_finish.disabled = True

    btn_finish.on_click(on_finish_clicked)
    
    # Display the button layout sequentially underneath the interactive widget canvas
    display(btn_finish, output_area)

In [6]:
def convert_to_3d(points_2d: np.ndarray, draw_time: np.ndarray) -> np.ndarray:
    """Convert 2D points with given drawing time to 3D

    Args:
        points_2d (Nx2 np.ndarray): _description_
        draw_time (Nx1 np.ndarray): _description_

    Returns:
        Nx3 np.ndarray: 3D points
    """
    z_range = np.mean(np.max(points_2d, axis=0) - np.min(points_2d, axis=0))
    z = np.sin(np.linspace(0, 4 * np.pi, points_2d.shape[0])) * z_range / 2
    points_3d = np.column_stack((points_2d[:, 0], points_2d[:, 1], z))
    return points_3d


In [7]:
def animate_curve(points_3d: np.ndarray, draw_time: np.ndarray, save_to_file: bool = True, allow_interaction: bool = True):
    """Animate a 3D curve by performing a few rotations about a changing axis

    Args:
        points_3d (np.ndarray): _description_
        draw_time (np.ndarray): _description_
        save_to_file (bool, optional): _description_. Defaults to True.
        allow_interaction (bool, optional): _description_. Defaults to True.

    Returns:
        _type_: _description_
    """
    num_pts = points_3d.shape[0]
    MS_PER_FRAME = 1000.0 / 24
    total_draw_time = draw_time[num_pts - 1] - draw_time[0]  
    draw_frames = int(total_draw_time / MS_PER_FRAME)  
    static_frames = 50
    anim_frames = 401
    total_frames = draw_frames + static_frames + anim_frames 
    frames_per_rotation = 40  

    # --- VisPy Scene Setup ---
    canvas = scene.SceneCanvas(keys='interactive', show=True, size=(512, 512), bgcolor='white', config={'samples': 4})
    canvas.show()

    view = canvas.central_widget.add_view()
    view.camera = scene.cameras.PanZoomCamera(rect=(-IMG_SIZE/2, -IMG_SIZE/2, IMG_SIZE, IMG_SIZE))
    view.camera.interactive = False

    # --- 1. Texture Background ---
    try:
        img_data = np.array(Image.open('paper_texture.png'))
    except FileNotFoundError:
        img_data = (np.random.rand(512, 512, 3) * 255).astype(np.uint8)

    image_visual = scene.visuals.Image(img_data, parent=view.scene)
    image_visual.transform = MatrixTransform()
    image_visual.transform.scale((IMG_SIZE / img_data.shape[1], IMG_SIZE / img_data.shape[0], 1))
    image_visual.transform.translate((-IMG_SIZE/2, -IMG_SIZE/2, 2)) 

    # --- 2. 3D Line ---
    line_visual = scene.visuals.LinePlot(data=points_3d, marker_size=5, edge_color='red', face_color='red', color='red', width=10, parent=view.scene)
    line_visual.transform = MatrixTransform()

    # 1. Manually set the scene graph drawing order (Higher number = drawn on top)
    image_visual.order = 1
    line_visual.order = 2

    # 2. Disable depth testing on the line so it never punches "behind" the image
    line_visual.set_gl_state(depth_test=False)  # pylint: disable=no-member

    # --- Recording Configuration ---
    frames_list = []  # Array to store raw pixel data from the GPU
    current_frame = 0

    def update_frame(_):
        nonlocal current_frame
        
        if current_frame >= total_frames:
            timer.stop()
            if save_to_file:
                print("Animation finished. Compiling video... please wait.")
                # Save frames using imageio (duration is in seconds per frame)
                imageio.mimsave('curve3d.mp4', frames_list, format='ffmpeg', fps=1000 // MS_PER_FRAME)
                print("Saved successfully as 'curve3d.mp4'!")
            if allow_interaction:
                # --- HAND OVER CONTROL TO MOUSE INTERACTION ---
                # 1. Safely clear the animation matrix from the line visual
                line_visual.transform.reset()
                
                # 2. Hide the background texture image
                image_visual.visible = False
                
                # 3. Instantiate the new 3D Turntable Camera with the correct orientation
                # fov=0 makes the camera orthographic
                view.camera = scene.cameras.TurntableCamera(parent=view.scene, elevation=90, azimuth=0, roll=0, fov=0)
                
                # 4. Set the camera's position and scale
                view.camera.center = (0, 0, 0)
                view.camera.scale_factor = IMG_SIZE
                
                # 5. Restore mouse event capture
                view.camera.interactive = True
                
                print("Canvas is now fully interactive! Click and drag to orbit in 3D.")
            else:
                # Close the canvas after the animation is finished
                canvas.close()
            return

        # --- Frame Logic ---
        if current_frame < draw_frames:
            cur_draw_time = current_frame * MS_PER_FRAME
            N = np.searchsorted(draw_time[:num_pts] - draw_time[0], cur_draw_time, side='right')
            if N > 1:
                line_visual.set_data(data=points_3d[:N])  # pylint: disable=no-member
                line_visual.visible = True
            else:
                line_visual.visible = False

        elif current_frame < draw_frames + static_frames:
            line_visual.set_data(data=points_3d)  # pylint: disable=no-member
            line_visual.visible = True
            line_visual.transform.reset()
            
        else:
            a_frame = current_frame - draw_frames - static_frames
            t = a_frame / (anim_frames - 1)

            if t < 0.5:
                t0 = t * 2
                axis = np.array([t0, 1.0 - t0, 0.0])
            else:
                t1 = (t - 0.5) * 2
                axis = np.array([1.0 - t1, -t1, 0.0])
            axis /= np.linalg.norm(axis)

            alpha = (a_frame % frames_per_rotation) / (frames_per_rotation - 1)
            angle = 360.0 * alpha

            tr = MatrixTransform()
            tr.rotate(angle, axis)
            line_visual.transform = tr

        canvas.update()

        if save_to_file:
            # CRITICAL: Force immediate rendering and grab pixel matrix from the GPU
            # This stores the frame into RAM for writing later
            # 1. Grab the raw 1024x1024 hardware pixels
            frame_pixels = canvas.render()
            
            # 2. Convert to a Pillow Image, resize it down to exactly 512x512, and turn back into an array
            resized_frame = np.array(Image.fromarray(frame_pixels).resize((512, 512), Image.Resampling.LANCZOS))
            
            # 3. Append the perfectly scaled frame
            frames_list.append(resized_frame)
        
        current_frame += 1

    # Setup and start the loop
    timer = app.Timer(interval=MS_PER_FRAME/1000.0, connect=update_frame, start=True)
    timer.start()

    return canvas, timer

In [8]:
CREATE_NEW_CURVE = False

if CREATE_NEW_CURVE:
    draw_curve_widget()
    
load_pts = np.load('curve_2d.npz')
pts_2d = load_pts['pts_2d']
draw_t = load_pts['draw_t']

In [9]:
%gui qt6
pts_3d = convert_to_3d(pts_2d, draw_t)
temp_canvas, temp_timer = animate_curve(pts_3d, draw_t)